# Neandertales Caffe — Class 2 (finish) + Class 3
## Today's plan:
1. **Finish Class 2** — revenue per quarter, bar chart per month, is_large_order, price_tier, challenges
2. **Class 3** — dimension tables, joins, order-level analysis, analytical layer, challenges

> Run the setup cell first — it loads everything from scratch.


# 0. Setup
Run this first — it loads the dataset and recreates all columns from last class.

In [ ]:
# ── Google Colab ───────────────────────────────────────────
# from google.colab import files
# files.upload()

# ── VS Code ─────────────────────────────────────────────────
# Make sure neandertales_orders.csv is in the same folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv("neandertales_orders.csv")

# Revenue column
orders["revenue"] = orders["price"] * orders["quantity"]

# Bag size
def extract_bag_size(product):
    if "500g" in product:
        return "500g"
    elif "1000g" in product:
        return "1000g"
    else:
        return "No size"

orders["bag_size"] = orders["product"].apply(extract_bag_size)

# Date columns
orders["date"]       = pd.to_datetime(orders["date"])
min_date             = orders["date"].min()
max_date             = orders["date"].max()

calendar             = pd.DataFrame({"date": pd.date_range(start=min_date, end=max_date, freq="D")})
calendar["year"]     = calendar["date"].dt.to_period("Y")
calendar["quarter"]  = calendar["date"].dt.to_period("Q")
calendar["year_month"]= calendar["date"].dt.to_period("M")
calendar["month_name"]= calendar["date"].dt.month_name()
calendar["day_name"] = calendar["date"].dt.day_name()
calendar["is_weekend"]= calendar["date"].dt.day_of_week >= 5

orders = orders.merge(calendar, on="date", how="left")

print("Dataset ready!")
print("Shape:", orders.shape)
print("Columns:", orders.columns.tolist())

---
# Finishing Class 2

### Your turn — revenue per quarter

In [ ]:
# Hint: df.groupby("col")["col"].sum()
rev_per_quarter = orders.groupby("_______")["_______"]._____().round(2)

print(rev_per_quarter)
print()
print("Q1 = Jan-Mar   Q2 = Apr-Jun   Q3 = Jul-Sep   Q4 = Oct-Dec")

### Your turn — bar chart of revenue per month

In [ ]:
# Step 1 — calculate
# Hint: df.groupby("col")["col"].sum()
rev_per_month = orders.groupby("_____")["_______"]._____().round(2)

# Step 2 — convert index for plotting
rev_per_month.index = rev_per_month.index.astype(str)

# Step 3 — plot
___________.plot(kind="bar", color="steelblue", figsize=(12, 4))

plt.title("Neandertales Caffe — Revenue per Month")
plt.xlabel("Month")
plt.ylabel("Revenue (euros)")
plt.xticks(rotation=___, ha="right")

plt.tight_layout()
plt.show()

## 2.3 — Two More Columns

### Your turn — create `price_tier` column
Low = below 15 euros, Mid = 15 to 35 euros, High = above 35 euros

In [ ]:
# Step 1 — write the function
# Hint: use if / elif / else with price thresholds
def classify_price(price):
    ____________________:
        return "Low"
    _________________ <= ___:
        return "Mid"
    ____________:
        return "High"



In [ ]:
# Step 2 — apply to every row
# Hint: df["col"].apply(function_name)
orders["price_tier"] = orders["_____"].apply(_______________)



In [ ]:
# Step 3 — check distribution
tier_counts = orders["price_tier"].value_counts()
print(tier_counts)

In [ ]:
# Hint: df.groupby("col")["col"].mean().sort_values(ascending=False)
avg_rev_tier = orders.groupby("__________")["_______"]._____().round(2)

avg_rev_tier_sorted = avg_rev_tier.sort_values(ascending=False)
print(avg_rev_tier_sorted)

# Challenge
One question — write everything yourself.

## Challenge 1 — Bar chart of revenue per bag size
Filter Coffee orders only, group by bag_size, sum revenue, plot a bar chart.
Add a title and axis labels.

**Hint:** filter → groupby → sum → plot

In [ ]:

# Challenge 1 — bar chart revenue per bag size (Coffee only)

# Step 1 — filter Coffee orders only
# Hint: df[df["col"] == "value"]
coffee = orders[orders["________"] == "______"]

print("Coffee orders:", len(coffee))

In [ ]:
# Step 2 — calculate revenue per bag size
# Hint: df.groupby("col")["col"].sum()
rev_by_size = 



In [ ]:

# Step 3 — sort highest first
# Hint: df.sort_values(ascending=False)
rev_by_size_sorted = 



In [ ]:
# Step 4 — print
print(rev_by_size_sorted.round(2))

In [ ]:
# Step 5 — plot a bar chart
# Hint: df.plot(kind="bar", color="steelblue", edgecolor="white")
___________.plot(kind="___", color="steelblue", edgecolor="white")

plt.title("___")
plt.xlabel("___")
plt.ylabel("___")
plt.xticks(rotation=___)

plt.tight_layout()
plt.show()

## Challeng 2 — Which quarter had the most orders?
Count the number of unique orders per quarter.
Print the result sorted highest first.

**Hint:** groupby quarter, nunique on order_id, sort

In [ ]:
# Your code here


# Answers — only open after you tried!

In [ ]:
# Challenge 1 — bar chart revenue per bag size (Coffee only)
coffee = orders[orders["category"] == "Coffee"]

rev_by_size = coffee.groupby("bag_size")["revenue"].sum().sort_values(ascending=False)

rev_by_size.plot(kind="bar", color="steelblue", edgecolor="white")

plt.title("Coffee Revenue — 500g vs 1000g")
plt.xlabel("Bag Size")
plt.ylabel("Total Revenue (euros)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print(rev_by_size.round(2))

In [ ]:
# Challenge 2 — orders per quarter
orders_per_quarter = orders.groupby("quarter")["order_id"].nunique()

orders_per_quarter_sorted = orders_per_quarter.sort_values(ascending=False)
orders_per_quarter.index  = orders_per_quarter.index.astype(str)

print("Orders per quarter:")
print(orders_per_quarter_sorted)

---
# Class 3 — From Transactional Data to Analytical Data
> *"Transactional data records what happened. Analytical data is shaped so you can ask why."*

# 3. Dimension Tables — Building a Proper Data Model
Right now our dataset is a **fact table** — every row is one transaction.

In real databases you split data into:
- **Fact table** — the transactions (what we have now)
- **Dimension tables** — reference data about products and customers

> *"A fact table records events. A dimension table describes the participants."*

## 3.1 — Product Dimension Table

In [ ]:
# Product dimension table — one row per product
# We join on product_id — NOT the product name
# Joining on IDs is safer — a typo in a name would break the join silently

products_dim = pd.DataFrame({
    "product_id": [
        "P001","P002","P003","P004","P005","P006","P007","P008",
        "P009","P010","P011","P012","P013","P014","P015"
    ],
    "product": [
        "Espresso Blend 500g",          "Espresso Blend 1000g",
        "Colombia Single Origin 500g",  "Colombia Single Origin 1000g",
        "Ethiopia Natural 500g",        "Ethiopia Natural 1000g",
        "Dark Roast Blend 500g",        "Dark Roast Blend 1000g",
        "Classic Mug 300ml",            "Travel Mug 400ml",
        "Filters V60 x100",             "Cleaning Tablets x10",
        "French Press 600ml",           "Moka Pot",
        "Coffee Grinder Manual"
    ],
    "category": [
        "Coffee","Coffee","Coffee","Coffee","Coffee","Coffee","Coffee","Coffee",
        "Accessories","Accessories","Accessories","Accessories",
        "Equipment","Equipment","Equipment"
    ],
    "bag_size": [
        "500g","1000g","500g","1000g","500g","1000g","500g","1000g",
        "No size","No size","No size","No size","No size","No size","No size"
    ],
    "roast": [
        "Medium","Medium","Light","Light",
        "Natural","Natural","Dark","Dark",
        None,None,None,None,None,None,None
    ],
    "origin": [
        "Blend","Blend","Colombia","Colombia",
        "Ethiopia","Ethiopia","Blend","Blend",
        None,None,None,None,None,None,None
    ],
    "base_price": [
        12.90,22.90,14.90,27.90,15.90,29.90,11.90,21.90,
        9.90,24.90,7.90,6.90,34.90,29.90,44.90
    ]
})

print("Product dimension table:")
print(products_dim.to_string(index=False))

### Your turn — how many products have a known roast?

In [ ]:
# Your turn — how many products have a known roast?
# Hint: df["col"].notna().sum()
known_roast = products_dim["_____"]._______().sum()

print("Products with a known roast:", known_roast)
print()

# Your turn — how many unique product_ids?
# Hint: df["col"].nunique()
unique_ids = products_dim["__________"].________()
print("Unique product IDs:", unique_ids)

## 3.2 — Customer Dimension Table

In [ ]:
# Customer dimension table — one row per customer
# Same customer_id values as the fact table — this is what makes the join work

customers_dim = pd.DataFrame({
    "customer_id": [
        "C001","C002","C003","C004","C005","C006","C007","C008","C009","C010",
        "C011","C012","C013","C014","C015","C016","C017","C018","C019","C020",
        "C021","C022","C023","C024","C025","C026","C027","C028","C029","C030"
    ],
    "customer": [
        "Alice","Bob","Carol","David","Eva","Frank","Grace","Henry","Iris","James",
        "Kate","Liam","Mia","Noah","Olivia","Paul","Quinn","Rosa","Sam","Tina",
        "Uma","Victor","Wendy","Xander","Yara","Zoe","Marco","Nina","Oscar","Paula"
    ],
    "city": [
        "Berlin","Berlin","Munich","Berlin","Cologne","Berlin","Hamburg","Munich",
        "Cologne","Berlin","Munich","Munich","Berlin","Berlin","Munich","Berlin",
        "Hamburg","Berlin","Munich","Munich","Cologne","Berlin","Berlin","Frankfurt",
        "Berlin","Cologne","Hamburg","Berlin","Frankfurt","Frankfurt"
    ],
    "email": [
        "alice@gmail.com","bob@outlook.com","carol@web.de","david@gmx.de",
        "eva@yahoo.com","frank@icloud.com","grace@gmail.com","henry@web.de",
        "iris@gmx.de","james@outlook.com","kate@gmail.com","liam@outlook.com",
        "mia@web.de","noah@gmx.de","olivia@yahoo.com","paul@icloud.com",
        "quinn@gmail.com","rosa@web.de","sam@gmx.de","tina@outlook.com",
        "uma@gmail.com","victor@outlook.com","wendy@web.de","xander@gmx.de",
        "yara@yahoo.com","zoe@icloud.com","marco@gmail.com","nina@web.de",
        "oscar@gmx.de","paula@outlook.com"
    ],
    "join_date": [
        "2023-02-14","2023-03-01","2023-01-20","2022-11-05","2023-05-18",
        "2022-08-30","2023-06-12","2022-12-01","2023-04-22","2022-09-15",
        "2023-07-08","2023-02-28","2022-10-17","2023-08-05","2022-07-19",
        "2023-09-11","2023-01-30","2022-06-25","2023-10-03","2022-05-14",
        "2023-11-20","2022-04-07","2023-12-01","2022-03-22","2024-01-09",
        "2022-02-18","2024-02-14","2022-01-05","2024-03-20","2023-03-15"
    ],
    "segment": [
        "Regular","Occasional","Regular","Regular","Occasional",
        "VIP","Regular","Occasional","Regular","VIP",
        "Regular","Regular","VIP","Occasional","Regular",
        "Occasional","Regular","VIP","Regular","Occasional",
        "Regular","Occasional","Regular","Regular","VIP",
        "Regular","Occasional","Regular","VIP","Regular"
    ],
    "preferred_coffee": [
        "Espresso Blend","Dark Roast Blend","Colombia Single Origin","Espresso Blend",
        "Ethiopia Natural","Colombia Single Origin","Dark Roast Blend","Espresso Blend",
        "Ethiopia Natural","Espresso Blend","Colombia Single Origin","Dark Roast Blend",
        "Espresso Blend","Ethiopia Natural","Colombia Single Origin","Espresso Blend",
        "Dark Roast Blend","Colombia Single Origin","Ethiopia Natural","Espresso Blend",
        "Colombia Single Origin","Dark Roast Blend","Espresso Blend","Ethiopia Natural",
        "Colombia Single Origin","Espresso Blend","Dark Roast Blend","Ethiopia Natural",
        "Espresso Blend","Colombia Single Origin"
    ]
})

# Convert join_date to datetime
customers_dim["join_date"] = pd.to_datetime(customers_dim["join_date"])

print("Customer dimension table:")
print(customers_dim.head(10).to_string(index=False))
print()
print(f"Total customers: {len(customers_dim)}")
print()
print("Segments:")
print(customers_dim["segment"].value_counts())
print()
print("Preferred coffee:")
print(customers_dim["preferred_coffee"].value_counts())

### Your turn — how many VIP customers do we have?

In [ ]:
# Hint: df[df["col"] == "value"]
vip_customers = _______[___________["_______"] == "______"]

vip_count = len(vip_customers)
print("VIP customers:", vip_count)

## 3.3 — Joining the Dimension Tables

In [ ]:
# Example — join orders with product dimension using product_id
# Why product_id and NOT product name?
# Because IDs are guaranteed unique
# A typo in a product name would break the join silently

# Check that product_id exists in both tables before joining
print("Fact table — product_id sample:")
print(orders["product_id"].head(5).tolist())
print()
print("Dimension — product_id sample:")
print(products_dim["product_id"].head(5).tolist())
print()

# Now join on product_id
orders_enriched = orders.merge(
    products_dim[["product_id", "roast", "origin", "bag_size", "base_price"]],
    on="product_id",
    how="left"
)

print("After joining product dimension:")
print(orders_enriched[["product_id", "product", "roast", "origin", "bag_size"]].head(8))

### Your turn — join orders_enriched with customer dimension

In [ ]:
# Your turn — join orders_enriched with customer dimension
# Hint: df.merge(other_df, on="shared_col", how="left")
# Use customer_id — always join on IDs, not names

orders_enriched = orders_enriched.merge(
    customers_dim[["customer_id", "segment", "preferred_coffee"]],
    on="___________",
    how="____"
)

print("Columns now available:")
print(orders_enriched.columns.tolist())
print()
print("Missing segment values:", orders_enriched["segment"].isnull().sum())

### Your turn — revenue per customer segment

In [ ]:
# Hint: df.groupby("col")["col"].sum().sort_values(ascending=False)
rev_by_segment = orders_enriched.groupby("_______")["_______"]._____().round(2)

rev_by_segment_sorted = rev_by_segment.sort_values(ascending=False)
print(rev_by_segment_sorted)

### Your turn — which coffee roast generates the most revenue?

In [ ]:
# Step 1 — filter Coffee only
# Hint: df[df["col"] == "value"]
coffee_e = orders_enriched[orders_enriched["________"] == "Coffee"]

# Step 2 — groupby roast, sum revenue
# Hint: df.groupby("col")["col"].sum().sort_values(ascending=False)
roast_rev = coffee_e.groupby("_____")["_______"]._____().round(2)

roast_rev_sorted = roast_rev.sort_values(ascending=False)
print(roast_rev_sorted)

## 4.0 — Order-Level Table

Right now `orders` has **one row per product per order**.  
That means an order with 3 products counts as 3 rows — so "average order value" or "orders per month" are misleading.

We need to **group by order** first to get a proper order-level view.

> This also fixes `is_large_order` — we should classify based on total items per order, not per row.


In [ ]:
# Build the order-level summary table
# Each row = one order, with total items, total revenue, number of different products

# Hint: df.groupby(["col1","col2",...]).agg(new_col=("col","function")).reset_index()
order_summary = orders.groupby(["order_id", "customer_id", "customer", "city", "year_month"]).agg(
    total_items   = ("quantity", "_____"),
    total_revenue = ("revenue",  "_____"),
    num_products  = ("product",  "_____"),
).round(2).reset_index()

print("Rows in orders:        ", len(orders))
print("Rows in order_summary: ", len(order_summary))
print()
print(order_summary.head(10))


### Fix `is_large_order` — classify at order level

Now that we have total items per order, we can classify correctly.


In [ ]:
# Step 1 — flag large orders based on total_items (not raw quantity)
# Hint: df["new_col"] = df["col"] >= value
order_summary["is_large_order"] = order_summary["_____"] >= 3

# Step 2 — count large orders
# Hint: boolean_series.sum() counts True values
total_orders  = len(order_summary)
large_orders  = order_summary["is_large_order"]._____
print("Large orders: ", large_orders)
print("Total orders: ", total_orders)

# Step 3 — percentage
# Hint: count / total * 100
large_pct = round(_____ / _____ * 100, 1)
print("Large orders:", large_pct, "% of all orders")

### Analysis on the order-level table


In [ ]:
# Average order value
# Hint: df["col"].mean()
avg_order_value = order_summary["_____"]._____()

print("Average order value:", round(avg_order_value, 2), "euros")


In [ ]:
# How many orders per month?
# Hint: df.groupby("col")["col"].count().reset_index()
orders_per_month = order_summary.groupby("_____")["order_id"]._____().reset_index()

orders_per_month["year_month"] = orders_per_month["year_month"].astype(str)
print(orders_per_month)


In [ ]:
# Your turn — plot orders per month as a line chart
# Hint: df.set_index("col")["col"].plot(kind="line", marker="o")
orders_per_month_plot = orders_per_month.set_index("_____")

orders_per_month_plot["_____"].plot(kind="line", marker="o", color="steelblue")

plt.title("_____")
plt.xlabel("Month")
plt.ylabel("_____")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Your turn — average order value per city
# Hint: df.groupby("col")["col"].mean().sort_values(ascending=False)
avg_order_by_city = order_summary.groupby("_____")["_____"]._____().round(2)

avg_order_by_city_sorted = avg_order_by_city.sort_values(ascending=False)
print(avg_order_by_city_sorted)


# 4. The Analytical Layer
Now that the data is enriched, we build summary tables
that could power a real dashboard.

> *"Transactional: one row per event.
> Analytical: one row per dimension you care about."*

## 4.1 — Monthly Summary Table

In [ ]:
# Example — build a monthly summary with multiple metrics at once
# agg() lets you calculate several things in one groupby
monthly_summary = orders_enriched.groupby("year_month").agg(
    total_revenue = ("revenue",  "sum"),
    total_orders  = ("order_id", "nunique"),
    total_units   = ("quantity", "sum"),
    avg_revenue   = ("revenue",  "mean")
).round(2)

# Convert index for display
monthly_summary.index = monthly_summary.index.astype(str)

print(monthly_summary)

### Your turn — build a city summary table
For each city show: total_revenue, total_orders, avg_revenue

In [ ]:
# Hint: same structure as above but groupby "city"
# df.groupby("col").agg(new_col=("col","function"), ...)
city_summary = orders_enriched.groupby("____").agg(
    total_revenue = ("_______", "___"),
    total_orders  = ("________", "______"),
    avg_revenue   = ("_______", "____")
).round(2)

city_summary_sorted = city_summary.sort_values("total_revenue", ascending=False)
print(city_summary_sorted)

---
# Challenges
Use `orders_enriched` which has all the new columns.

## Challenge 1 — Revenue per customer segment, bar chart
Which segment generates the most revenue? Show it as a bar chart.

**Hint:** groupby `segment`, sum `revenue`, plot

In [ ]:
# Challenge 1 — revenue per segment bar chart
rev_by_segment = _______________.____________("__________")["___________"].__________().____________(ascending=False).round(2)

______________.plot(kind="bar", color="steelblue", edgecolor="white")

plt.title("Revenue per Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Total Revenue (euros)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print(rev_by_segment)


# Answers — only open after you have tried!

In [ ]:
# Challenge 1 — revenue per segment bar chart
rev_by_segment = orders_enriched.groupby("segment")["revenue"].sum().sort_values(ascending=False).round(2)

rev_by_segment.plot(kind="bar", color="steelblue", edgecolor="white")

plt.title("Revenue per Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Total Revenue (euros)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print(rev_by_segment)
